# Exercise 8
Building an mnist voting classifier.

In [1]:
from sklearn.datasets import fetch_openml
import numpy as np
mnist = fetch_openml('mnist_784', version = 1)
X, y = mnist['data'], mnist['target']
X = X.to_numpy()
y = y.to_numpy().astype(np.uint8)

In [2]:
X_train, X_val, X_test = X[:50000], X[50000:60000], X[60000:]
y_train, y_val, y_test = y[:50000], y[50000:60000], y[60000:]

In [3]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

rndf_clf = RandomForestClassifier(n_estimators = 100, n_jobs = -1)
ext_clf = ExtraTreesClassifier(n_estimators = 300, n_jobs = -1)
svm_clf = SVC(max_iter = -1, probability = True, kernel = 'rbf')
log_clf = LogisticRegression(n_jobs = -1)
voting_clf = VotingClassifier(
    [('random forest', rndf_clf),('extra trees', ext_clf),('svm classifier', svm_clf),('logistic regression', log_clf)],
    voting = 'soft',
    n_jobs = -1,
    weights = [0.2, 0.2, 0.4, 0.2]
)


for clf in (rndf_clf, ext_clf, svm_clf, log_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    print(clf.__class__.__name__, accuracy_score(y_val, y_pred))


RandomForestClassifier 0.9716
ExtraTreesClassifier 0.9759
SVC 0.9802


/home/thoalfeqar/miniconda3/envs/tf/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression 0.9272


/home/thoalfeqar/miniconda3/envs/tf/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


VotingClassifier 0.9784


In [5]:
for clf in (rndf_clf, ext_clf, svm_clf, log_clf, voting_clf):
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))


RandomForestClassifier 0.9677
ExtraTreesClassifier 0.9732
SVC 0.9785
LogisticRegression 0.9243
VotingClassifier 0.9753


# Stacking
Building a stacking ensemble model

In [30]:
def get_predictions(clfs, dataset):
    predictions = np.zeros(shape = (dataset.shape[0], len(clfs) * 10), dtype = float)
    for i, clf in enumerate(clfs):
        y_pred = clf.predict_proba(dataset)
        predictions[:, i*10:(i+1)*10] = y_pred
    return predictions

In [31]:
val_predictions = get_predictions((rndf_clf, ext_clf, svm_clf, log_clf, voting_clf), X_val)

In [33]:
svm_blender = SVC(kernel = 'rbf')
svm_blender.fit(val_predictions, y_val)

SVC()

In [35]:
test_predictions = get_predictions((rndf_clf, ext_clf, svm_clf, log_clf, voting_clf), X_test)

In [36]:
y_pred = svm_blender.predict(test_predictions)
accuracy_score(y_test, y_pred)

0.9783